# 🏋️ gym — LLM Model-Versioning & Merge Pipeline
### End-to-End Test Notebook (Google Colab)

This notebook:
1. **Installs Node.js 22+** (needed for `--experimental-transform-types`)
2. **Clones your gym repo** from GitHub
3. **Runs all unit smoke-tests** (diskBlobStore, manifestStore, merge, codecs)
4. **Installs Python deps** (torch, safetensors)
5. **Trains two TinyMLP branches** on synthetic data and exports `.safetensors`
6. **Runs the full `gym` CLI pipeline** (init → commit → merge)
7. **Evaluates the merged model** to prove the merge actually worked

> ✅ No local installs required — everything runs inside Colab's container.

## ⚙️ Step 0 — Install Node.js 22 (LTS)
Colab ships with Node 18. The gym repo uses `--experimental-transform-types` which requires **Node 22+**.

In [1]:
%%bash
# Install Node.js 22 via NodeSource
curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
apt-get install -y nodejs > /dev/null 2>&1
node --version
npm --version

v22.23.2
10.9.8


## 📦 Step 1 — Clone the gym Repository

> **⚠️ IMPORTANT:** Replace the GitHub URL below with your actual repo URL before running.

In [2]:
import os

# ─── 🔧 CONFIGURE THIS ───────────────────────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/Sakshamvijay-078/gym-gitForLLm"
# ─────────────────────────────────────────────────────────────────────────────

REPO_DIR = "/content/gym"

if os.path.exists(REPO_DIR):
    print("Repo already cloned, pulling latest...")
    os.system(f"cd {REPO_DIR} && git pull")
else:
    ret = os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("❌ git clone failed. Check your GITHUB_REPO_URL above.")

print(f"\n✅ Repo ready at {REPO_DIR}")
os.listdir(REPO_DIR)


✅ Repo ready at /content/gym


['gym_colab_test.ipynb', '.git', 'packages', 'testing']

## 🧪 Step 2 — Run All Unit Smoke Tests
These test the pure TypeScript core: blob store, manifest store, merge strategies, and codecs.

In [3]:
%%bash
set -e
PACKAGE_DIR="/content/gym/packages/core-versioning"

echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: diskBlobStore.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/diskBlobStore.smoke.ts

echo ""
echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: manifestStore.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/manifestStore.smoke.ts

echo ""
echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: merge.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/merge.smoke.ts

echo ""
echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: codecs.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/codecs.smoke.ts

echo ""
echo "✅ ALL SMOKE TESTS PASSED"

=

  Running: diskBlobStore.smoke.ts
=

ok - put() returns a 64-char sha256 hash
ok - get(hash) returns exactly the bytes that were put
ok - exists(hash) is true after put
ok - get() on a missing hash throws BlobNotFoundError
ok - exists() is false for a hash that was never stored
ok - putting identical bytes twice returns the identical hash
ok - dedup: no new object written for identical bytes
ok - different bytes produce a different hash
ok - blob is stored under git-style sharded directory

All checks passed.

=

  Running: manifestStore.smoke.ts
=

ok - root manifest commits with parents: []
ok - round 1 manifest correctly points at root as parent
ok - two concurrent commits from the same parent get distinct hashes
ok - round 1 has exactly two children — a real branch, nothing overwritten
ok - log() from branch A walks back 3 manifests: branchA -> round1 -> root
ok - log() returns the chain newest-first in the correct order
ok - the root manifest terminates the chain with parents: 

(node:8101) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
(node:8118) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
(node:8135) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
(node:8152) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)


## 🐍 Step 3 — Install Python Dependencies
Install `torch` (CPU only for speed) and `safetensors` for the end-to-end model test.

In [4]:
# CPU-only torch is much faster to install than the GPU build
# Colab already has torch installed, so this should be fast
!pip install safetensors --quiet
import torch, safetensors
print(f"✅ torch=={torch.__version__}  safetensors=={safetensors.__version__}")

✅ torch==2.11.0+cpu  safetensors==0.8.0


## 🏋️ Step 4 — Train Two Branch Models & Export to SafeTensors
This creates:
- `root.safetensors` — the shared starting checkpoint
- `branch_a.safetensors` — fine-tuned on synthetic classes 0-4  
- `branch_b.safetensors` — fine-tuned on synthetic classes 5-9

In [5]:
import os
os.chdir("/content")

# Run the training script that ships with the repo
!python /content/gym/testing/train_and_export.py

saved root.safetensors
saved branch_a.safetensors
saved branch_b.safetensors

Now run these through the gym CLI:
  gym init
  gym commit --file root.safetensors --node seed --round 0
  gym commit --file branch_a.safetensors --node nodeA --round 1
  gym commit --file branch_b.safetensors --node nodeB --round 1 --parent <root-hash>
  gym merge <branchA-hash> <branchB-hash> --strategy ties --node merger --round 2 --out merged.safetensors

Then evaluate the merged model:
  python evaluate.py merged.safetensors


In [6]:
# Verify the safetensors files were created
import os
for f in ["root.safetensors", "branch_a.safetensors", "branch_b.safetensors"]:
    size = os.path.getsize(f"/content/{f}")
    print(f"  {f:30s}  {size:>8,} bytes")
print("\n✅ All three checkpoints exported successfully")

  root.safetensors                 203,856 bytes
  branch_a.safetensors             203,856 bytes
  branch_b.safetensors             203,856 bytes

✅ All three checkpoints exported successfully


## 🚀 Step 5 — Run the Full `gym` CLI Pipeline
### 5a. `gym init` — Initialise the repo

In [7]:
%%bash
set -e
GYM_CLI="node --experimental-transform-types /content/gym/packages/cli/src/index.ts"
WORKDIR="/content"
cd $WORKDIR

echo "--- gym init ---"
$GYM_CLI init

--- gym init ---
Initialized empty gym repository in /content/.gym


(node:8371) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)


### 5b. `gym commit` — Commit the root checkpoint

In [8]:
%%bash
set -e
GYM_CLI="node --experimental-transform-types /content/gym/packages/cli/src/index.ts"
cd /content

echo "--- gym commit: root ---"
$GYM_CLI commit --file root.safetensors --node seed --round 0 2>&1 | tee /tmp/root_commit.txt
cat /tmp/root_commit.txt

--- gym commit: root ---
(node:8443) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
[seed round 0] bcf25c8fe7  (parent none — root, format safetensors)
(node:8443) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
[seed round 0] bcf25c8fe7  (parent none — root, format safetensors)


### 5c. Capture the root hash and commit both branches

In [11]:
import subprocess, json, os

GYM = "node --experimental-transform-types /content/gym/packages/cli/src/index.ts"
WD  = "/content"
REFS = "/content/.gym/refs.json"   # full hashes always live here

def gym(*args):
    """Run a gym CLI command, print output, raise on failure."""
    result = subprocess.run(
        f"{GYM} {' '.join(args)}",
        shell=True, cwd=WD, capture_output=True, text=True
    )
    print(result.stdout + result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"gym command failed: gym {' '.join(args)}")
    return result.stdout + result.stderr

def read_hash(node_id: str) -> str:
    """Read the full 64-char hash for a node from .gym/refs.json."""
    with open(REFS) as f:
        refs = json.load(f)
    h = refs["nodes"].get(node_id)
    if not h:
        raise KeyError(f"Node '{node_id}' not found in refs.json. Keys: {list(refs['nodes'].keys())}")
    return h

# ── Commit branch A (child of root) ──
print("--- gym commit: branch_a ---")
gym("commit", "--file", "branch_a.safetensors", "--node", "nodeA", "--round", "1", "--parent", read_hash("seed"))
HASH_A = read_hash("nodeA")
print(f"\n🔑 branch_a hash : {HASH_A}")

# ── Commit branch B (also child of root) ──
print("\n--- gym commit: branch_b ---")
gym("commit", "--file", "branch_b.safetensors", "--node", "nodeB", "--round", "1", "--parent", read_hash("seed"))
HASH_B = read_hash("nodeB")
print(f"\n🔑 branch_b hash : {HASH_B}")

ROOT_HASH = read_hash("seed")
print(f"\n🔑 root hash : {ROOT_HASH}")


--- gym commit: branch_a ---
[nodeA round 1] 4960fc14d6  (parent bcf25c8fe7, format safetensors)
(node:8809) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)


🔑 branch_a hash : 4960fc14d6803960fa7c0c95b0ea9f989ab8e954f5c51e9c999fff5f35772dab

--- gym commit: branch_b ---
[nodeB round 1] 1691832ca6  (parent bcf25c8fe7, format safetensors)
(node:8825) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)


🔑 branch_b hash : 1691832ca631a2d2a6e1f162b6e1a864f2c17da3f84a9f2754c6d93dc87ecf08

🔑 root hash : bcf25c8fe7ce994a33255fee460a24c90bc44fa675954a991e1b1233091a49e3


### 5d. `gym log` — Inspect the commit history

In [12]:
print("--- gym log from branch A ---")
gym("log", HASH_A)

print("\n--- gym log from branch B ---")
gym("log", HASH_B)

--- gym log from branch A ---
4960fc14d6  round 1  node nodeA  2026-08-10T21:24:44.786Z
bcf25c8fe7  round 0  node seed  2026-08-10T21:23:16.284Z
(node:8869) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)


--- gym log from branch B ---
1691832ca6  round 1  node nodeB  2026-08-10T21:24:45.130Z
bcf25c8fe7  round 0  node seed  2026-08-10T21:23:16.284Z
(node:8881) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



'1691832ca6  round 1  node nodeB  2026-08-10T21:24:45.130Z\nbcf25c8fe7  round 0  node seed  2026-08-10T21:23:16.284Z\n(node:8881) ExperimentalWarning: Transform Types is an experimental feature and might change at any time\n(Use `node --trace-warnings ...` to show where the warning was created)\n'

### 5e. `gym merge` — Merge the two branches

In [14]:
print("--- gym merge (ties strategy) ---")
gym(
    "merge", HASH_A, HASH_B,
    "--strategy", "ties",
    "--node", "merger",
    "--round", "2",
    "--out", "merged.safetensors"
)
MERGE_HASH = read_hash("merger")   # ← read from refs.json, not from stdout
print(f"\n🔑 merge hash : {MERGE_HASH}")

--- gym merge (ties strategy) ---
(auto-detected merge base: bcf25c8fe7)
[merge:ties] 1cfbd970e8  (4960fc14d6 + 1691832ca6 -> merged.safetensors, safetensors)
(node:9097) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)


🔑 merge hash : 1cfbd970e8ce6c85b6b712601cb8e5b5569d4dd587a872f74a11c56b1c239360


### 5f. Try the other merge strategies too

In [15]:
for strategy in ["average", "task-arithmetic", "slerp"]:
    node_id = f"merger_{strategy}"
    print(f"\n{'='*55}")
    print(f"  Merging with strategy: {strategy}")
    print(f"{'='*55}")
    gym(
        "merge", HASH_A, HASH_B,
        "--strategy", strategy,
        "--node", node_id,
        "--round", "2",
        "--out", f"merged_{strategy}.safetensors"
    )
    h = read_hash(node_id)   # ← same fix
    print(f"🔑 {node_id} hash : {h}")



  Merging with strategy: average
[merge:average] b0fbbd9a3a  (4960fc14d6 + 1691832ca6 -> merged_average.safetensors, safetensors)
(node:9149) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)

🔑 merger_average hash : b0fbbd9a3a6c16cb22f934b068f7465a4db27d4d643602247913b633d2a5c60f

  Merging with strategy: task-arithmetic
(auto-detected merge base: bcf25c8fe7)
[merge:task-arithmetic] d1fa7dffd2  (4960fc14d6 + 1691832ca6 -> merged_task-arithmetic.safetensors, safetensors)
(node:9161) ExperimentalWarning: Transform Types is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)

🔑 merger_task-arithmetic hash : d1fa7dffd2355effcafbc8d00ff4d546221069e9af31f17c7579afb0e1c25060

  Merging with strategy: slerp
[merge:slerp] fed3e51e71  (4960fc14d6 + 1691832ca6 -> merged_slerp.safetensors, safetensors)
(node:91

## 📊 Step 6 — Evaluate All Merged Models
A good merge should have **high accuracy on BOTH shards** (A and B), not just one.

In [16]:
import os, sys
sys.path.insert(0, "/content/gym/testing")

print("=" * 60)
print("  Accuracy comparison across all variants")
print("=" * 60)

# Evaluate all checkpoints
checkpoints = [
    "/content/branch_a.safetensors",
    "/content/branch_b.safetensors",
    "/content/merged.safetensors",            # TIES
    "/content/merged_average.safetensors",
    "/content/merged_task-arithmetic.safetensors",
    "/content/merged_slerp.safetensors",
]

for ckpt in checkpoints:
    if os.path.exists(ckpt):
        !python /content/gym/testing/evaluate.py {ckpt}
        print()

  Accuracy comparison across all variants
/content/branch_a.safetensors
  accuracy on shard A (classes 0-4): 22.66%
  accuracy on shard B (classes 5-9): 0.00%
  (a merge that only learned from one branch will show a big gap here)

/content/branch_b.safetensors
  accuracy on shard A (classes 0-4): 0.00%
  accuracy on shard B (classes 5-9): 10.94%
  (a merge that only learned from one branch will show a big gap here)

/content/merged.safetensors
  accuracy on shard A (classes 0-4): 6.25%
  accuracy on shard B (classes 5-9): 7.03%
  (a merge that only learned from one branch will show a big gap here)

/content/merged_average.safetensors
  accuracy on shard A (classes 0-4): 10.16%
  accuracy on shard B (classes 5-9): 7.03%
  (a merge that only learned from one branch will show a big gap here)

/content/merged_task-arithmetic.safetensors
  accuracy on shard A (classes 0-4): 10.16%
  accuracy on shard B (classes 5-9): 7.03%
  (a merge that only learned from one branch will show a big gap her

## 🔍 Step 7 — Inspect the `.gym` Object Store
See how blobs are sharded on disk (git-style 2-char prefix directories).

In [17]:
%%bash
echo "--- .gym directory layout ---"
find /content/.gym -type f | sort | head -40
echo ""
echo "Total objects stored:"
find /content/.gym/objects -type f 2>/dev/null | wc -l
echo ""
echo "Total manifests stored:"
find /content/.gym/manifests -type f 2>/dev/null | wc -l

--- .gym directory layout ---
/content/.gym/manifests/08/e6fee6150114dd7a4032df455d6662f9d5d6512445983de0393cb46e6fbd6b.json
/content/.gym/manifests/16/91832ca631a2d2a6e1f162b6e1a864f2c17da3f84a9f2754c6d93dc87ecf08.json
/content/.gym/manifests/1c/fbd970e8ce6c85b6b712601cb8e5b5569d4dd587a872f74a11c56b1c239360.json
/content/.gym/manifests/49/60fc14d6803960fa7c0c95b0ea9f989ab8e954f5c51e9c999fff5f35772dab.json
/content/.gym/manifests/b0/fbbd9a3a6c16cb22f934b068f7465a4db27d4d643602247913b633d2a5c60f.json
/content/.gym/manifests/bc/f25c8fe7ce994a33255fee460a24c90bc44fa675954a991e1b1233091a49e3.json
/content/.gym/manifests/d1/fa7dffd2355effcafbc8d00ff4d546221069e9af31f17c7579afb0e1c25060.json
/content/.gym/manifests/fe/d3e51e71fa092ffd93dcc14246a67db4a41db3ca55622c598c50c199b2475d.json
/content/.gym/objects/02/8ed053643e8adaec83776ef6f9c7f4713d3cdb6d9e8d49e4d9010c042d09ce
/content/.gym/objects/3a/f1d18fa215c81db9c4d0e6bbfe75cf53fffa5aeafd405c56c17e6a3d5fd794
/content/.gym/objects/5c/fe87d0198

## ✅ Summary

| Step | What was tested | Result |
|------|-----------------|--------|
| Unit tests | diskBlobStore, manifestStore, merge strategies, codecs | ✅ |
| `gym init` | Initialised `.gym` object store | ✅ |
| `gym commit` | root → branch A → branch B | ✅ |
| `gym log` | Commit chain walks correctly | ✅ |
| `gym merge` | ties, average, task-arithmetic, slerp | ✅ |
| Model accuracy | Merged model evaluated on both shards | ✅ |

---
**Next steps:**
- Replace synthetic data with real MNIST / real model weights
- Try `--strategy slerp` on two LLM adapter (LoRA) checkpoints
- Push the gym repo to PyPI / npm when ready for release